# TD 2 — Local models 🦙

In this lab, we will download a model, run it **on our own machine** and use it from Python.

**At the end of this lab, you will know how to:**

- install and query a local model with Ollama;
- see the tokens and measure the speed;
- control the generation (temperature, seed);
- estimate the memory a model needs;
- get a structured answer;
- give a tool to a model.

Why a local model? Your data stays with you, it works without internet, it is free to use and it does not change without warning. In exchange, it is smaller and often less good than a hosted model.

## 1. Install Ollama 📦

Ollama is a tool that runs LLMs on your machine. It provides a command-line interface and a local HTTP API.

1. Install the version for your operating system from the [official website](https://ollama.com/download/).
2. Check the installation in a terminal (PowerShell on Windows):

```bash
ollama --version
```

3. Ollama must be running in the background: open `http://localhost:11434` in your browser, the message "Ollama is running" should appear.

## 2. Choose a model 🤖

The available models are in the [Ollama library](https://ollama.com/library). We need a small **general-purpose** model, able to chat and to use tools:

| Model | Download size | For which machine? |
| --- | --- | --- |
| `qwen2.5:1.5b` | about 1 GB | modest laptop (8 GB of memory) |
| `qwen2.5:3b` | about 1.9 GB | **default choice in this lab** |
| `ministral-3:3b` | about 3 GB | French model (Mistral), 16 GB recommended |

In a terminal, download the model you chose, then check:

```bash
ollama pull qwen2.5:3b
ollama list
```

> The part after the colon is the **tag**. Without a tag, Ollama uses `latest`. Always write the tag, so that you know exactly which model you are using.

## 3. Chat in the terminal ▶️

```bash
ollama run qwen2.5:3b
```

Ask it a few questions, then quit with `/bye`. Also ask it a question about your town: small models easily **make things up**.

## 4. The HTTP API 🔌

Ollama is a **web server**. Any program that can send an HTTP request can use it. Let's check that the server answers:

In [ ]:
import requests

BASE_URL = "http://localhost:11434"  # Ollama's default port
MODEL = "qwen2.5:3b"                 # change it if you chose another model

try:
    response = requests.get(BASE_URL, timeout=3)
    print(response.status_code, response.text)
except requests.ConnectionError:
    print("Ollama is not running. Start the app, or run `ollama serve` in a terminal.")

The `/api/tags` endpoint lists the installed models, like `ollama list`:

In [ ]:
installed = requests.get(f"{BASE_URL}/api/tags", timeout=10).json()

for model in installed["models"]:
    details = model["details"]
    print(f'{model["name"]:25s} {model["size"] / 1e9:5.1f} GB  {details["parameter_size"]:>8s}  {details["quantization_level"]}')

### 4.1 Generate an answer ✍️

`/api/generate` takes a prompt and returns the answer. With `"stream": False`, we receive everything at once. `raise_for_status()` raises an error if the request failed, for example if the model is not downloaded.

In [ ]:
payload = {"model": MODEL, "prompt": "Hello", "stream": False}

response = requests.post(f"{BASE_URL}/api/generate", json=payload, timeout=300)
response.raise_for_status()
data = response.json()

print(data["response"])

In [ ]:
data

### 4.2 See the tokens 🔍

The `context` field contains the **identifiers of the tokens** read and generated by the model. With Qwen 2.5, we recognise:

| Identifier | Text | Role |
| --- | --- | --- |
| `151644` | `<\|im_start\|>` | start of a message |
| `8948` | `system` | message from the system |
| `872` | `user` | message from the user |
| `77091` | `assistant` | message from the model |
| `198` | new line | separator |
| `9707` | `Hello` | your prompt |
| `151645` | `<\|im_end\|>` | end of a message |

You wrote only one word, but the model read many more: Ollama adds a **template**, which contains a default system message here.

### Demonstration: measure the speed ⏱️

From `data`, the `show_stats` function displays the number of tokens read (`prompt_eval_count`) and generated (`eval_count`), the speed and the loading time. Durations are in **nanoseconds**. We then send the request a second time.

For reference, 10 to 20 tokens per second is already comfortable to read.

In [ ]:
def show_stats(data):
    speed = data["eval_count"] / (data["eval_duration"] / 1e9)
    print(f'Tokens read      : {data["prompt_eval_count"]}')
    print(f'Tokens generated : {data["eval_count"]}')
    print(f'Speed            : {speed:.1f} tokens/s')
    print(f'Loading time     : {data["load_duration"] / 1e9:.2f} s')

show_stats(data)

# Second request
response = requests.post(f"{BASE_URL}/api/generate", json=payload, timeout=300)
response.raise_for_status()
show_stats(response.json())

For the second request, the model is already in memory: loading is almost immediate.

## 5. A conversation 💬

`/api/chat` takes a **list of messages**. Each message has a role:

- `system`: the general instructions, at the beginning;
- `user`: what the user says;
- `assistant`: what the model answered;
- `tool`: the result of a tool (part 10).

In [ ]:
def ask(messages, **options):
    """Send a conversation to the model and return the text of its answer."""
    payload = {"model": MODEL, "messages": messages, "stream": False, "options": options}
    response = requests.post(f"{BASE_URL}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    return response.json()["message"]["content"]


messages = [
    {"role": "system", "content": "You are a concise teaching assistant. Answer in at most two sentences."},
    {"role": "user", "content": "What is a token in a language model?"},
]
print(ask(messages))

> Is the answer right? Compare it with the lecture: a token is a word or a **piece of a word**. A small model can be confidently wrong.

The server **does not remember** previous requests: for a second question to make sense, you must send the whole conversation again.

### Demonstration: with and without history 🧾

1. Two separate conversations: the model cannot know the name.
2. A **single conversation**: we add the model's answer to the list (role `assistant`), then the next question.

In [ ]:
# 1. Two separate conversations
ask([{"role": "user", "content": "My name is Ada."}])
print("Without history:", ask([{"role": "user", "content": "What is my name?"}]))

# 2. One conversation
history = [{"role": "user", "content": "My name is Ada."}]
history.append({"role": "assistant", "content": ask(history)})
history.append({"role": "user", "content": "What is my name?"})
print("With history   :", ask(history))

## 6. Control the generation ⚙️

| Option | Meaning |
| --- | --- |
| `temperature` | randomness: `0` = always the most probable token (almost deterministic); higher = more varied |
| `seed` | fixes the randomness, to reproduce an answer |
| `num_predict` | maximum number of tokens to generate |
| `num_ctx` | size of the context window |

Let's change **only** the temperature:

In [ ]:
question = [{"role": "user", "content": "Invent the name of a bakery in Marseille. Answer with the name only."}]

for temperature in (0, 1, 2):
    answers = [ask(question, temperature=temperature, num_predict=30).strip() for _ in range(4)]
    print(f"temperature={temperature}: {len(set(answers))} different answers out of 4 -> {answers}")

### Demonstration: the seed 🌱

Let's keep `temperature=1` and count the different answers out of four attempts: without a seed, with `seed=42` each time, then with seeds 1, 2, 3 and 4.

In [ ]:
def count(label, list_of_options):
    answers = [ask(question, temperature=1, num_predict=30, **o).strip() for o in list_of_options]
    print(f"{label:14s} {len(set(answers))} different -> {answers}")

count("no seed", [{}] * 4)
count("seed=42", [{"seed": 42}] * 4)
count("seeds 1 to 4", [{"seed": s} for s in (1, 2, 3, 4)])

On a local model, the same seed and the same settings generally give the same answer again. Another seed gives another draw.

**Out of curiosity:** what happens at an absurd temperature? All the probabilities become almost equal. Yet the first answer below still makes sense: by default, Ollama only keeps the most probable tokens (`top_k`, `top_p`). In the second one, we remove this filter: the model draws at random from its **whole** vocabulary.

In [ ]:
print(ask(question, temperature=10**20, num_predict=30))
print("---")
print(ask(question, temperature=10**20, top_k=0, top_p=1.0, min_p=0.0, num_predict=30))

## 7. Streaming 📡

With `"stream": True`, the server sends the answer **piece by piece**: one JSON object per line. This is how chat interfaces display text progressively.

In [ ]:
import json

def stream_answer(prompt):
    payload = {"model": MODEL, "prompt": prompt, "stream": True}
    with requests.post(f"{BASE_URL}/api/generate", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line)
                print(chunk["response"], end="", flush=True)
    print()

stream_answer("Give three short reasons to run a language model locally.")

### Demonstration: the same with a conversation 💬

The `stream_chat(messages)` function uses `/api/chat`. This time, the text is in `message.content`. The function displays it progressively and also **returns** the complete text.

In [ ]:
def stream_chat(messages):
    payload = {"model": MODEL, "messages": messages, "stream": True}
    text = ""
    with requests.post(f"{BASE_URL}/api/chat", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line:
                piece = json.loads(line)["message"]["content"]
                print(piece, end="", flush=True)
                text += piece
    print()
    return text

answer = stream_chat([{"role": "user", "content": "Describe Provence in one sentence."}])
print(len(answer), "characters")

## 8. How much memory is needed? 💾

A model is a long list of numbers, its **parameters**. The memory needed depends on how many there are and on their precision:

$$\text{size in bytes} \approx \text{number of parameters} \times \frac{\text{bits per parameter}}{8}$$

**Quantization** reduces the number of bits (16, 8, 4...). The model is smaller and faster, but slightly less accurate. This is why a model with 7 billion parameters can fit on a laptop.

### Demonstration: estimate the size of a model 📐

Let's compute the size of a model with 7 billion parameters in 16, 8 and 4 bits. Then let's query `/api/show` to get the number of parameters of **our** model, and compare the estimate with the real size given by `/api/tags`.

In [ ]:
def size_gb(parameters, bits):
    return parameters * bits / 8 / 1e9

for bits in (16, 8, 4):
    print(f"7B in {bits:2d} bits: {size_gb(7e9, bits):.1f} GB")

info = requests.post(f"{BASE_URL}/api/show", json={"model": MODEL}, timeout=30).json()
parameters = info["model_info"]["general.parameter_count"]
print(info["details"]["parameter_size"], info["details"]["quantization_level"])
print(f"Estimate in 4 bits: {size_gb(parameters, 4):.2f} GB")

real = next(m["size"] for m in installed["models"] if m["name"] == MODEL)
print(f"Real size         : {real / 1e9:.2f} GB")

The real file is larger than the estimate: the `Q4_K_M` quantization keeps some parameters with more than 4 bits.

## 9. The `ollama` library 🐍

The official library hides the details (addresses, JSON) and adds convenient features.

**With pip**: change the next cell from "raw" to "code". **With uv**: `uv add ollama pydantic` in a terminal.

In [ ]:
import ollama

response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "Why is the sky blue? Answer in one sentence."}],
)
print(response.message.content)

> Ollama also provides an API compatible with OpenAI's, at `http://localhost:11434/v1`. Many tools can therefore use your local model without any change.

### Structured output 🧩

As in TD 1, we can force a schema. This is very useful to **extract information** from a text.

In [ ]:
from pydantic import BaseModel

class Book(BaseModel):
    title: str
    author: str
    year: int

text = """The Lord of the Rings is a fantasy novel written by J.R.R. Tolkien.
When Gandalf falls into the abyss with the Balrog, he says "Fly, you fools!" to the Fellowship.
The first volume was published in 1954."""

response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the book information from this text:\n{text}"}],
    format=Book.model_json_schema(),
    options={"temperature": 0},
)

book = Book.model_validate_json(response.message.content)
print(book)

### Demonstration: form is not truth 🎭

The schema guarantees the **form**, not the **truth**. Let's remove the date from the text:

1. with the `Book` schema, the year is required: the model must write one, from memory or at random;
2. with `BookOrNull`, the year may be missing, and the prompt says so.

In [ ]:
text_without_date = "The Lord of the Rings is a fantasy novel written by J.R.R. Tolkien."

def extract(schema, instruction):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": f"{instruction}\n\n{text_without_date}"}],
        format=schema.model_json_schema(),
        options={"temperature": 0},
    )
    return schema.model_validate_json(response.message.content)

# 1. The schema requires a year: the model must write one, from its memory or at random
print(extract(Book, "Extract the book information from this text:"))

# 2. The year may be missing
class BookOrNull(BaseModel):
    title: str
    author: str
    year: int | None

print(extract(BookOrNull, "Extract the book information. Use only this text. If the year is not written in the text, year must be null."))

Even with a schema that allows missing information, check: a small model does not always obey.

## 10. Give a tool to the model 🛠️

A model can only produce text. To act, it **asks** to use a tool, and **your program** runs it:

1. you send the question and the **description** of the tools;
2. the model answers with a request: `response.message.tool_calls`;
3. your program runs the function;
4. you send the result back with the `tool` role;
5. the model writes the final answer.

In [ ]:
def multiply(a, b):
    return a * b

multiply_description = {
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers and return the exact result",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "First number"},
                "b": {"type": "number", "description": "Second number"},
            },
            "required": ["a", "b"],
        },
    },
}

# Steps 1 and 2: the model asks to use the tool
messages = [{"role": "user", "content": "How much is 4827 multiplied by 3916?"}]
response = ollama.chat(model=MODEL, messages=messages, tools=[multiply_description], options={"temperature": 0})

print(response.message.tool_calls)

In [ ]:
# Steps 3, 4 and 5
if response.message.tool_calls:
    messages.append(response.message)                       # the request of the model
    for call in response.message.tool_calls:
        result = multiply(**call.function.arguments)        # our program runs the function
        messages.append({"role": "tool", "tool_name": call.function.name, "content": str(result)})
    final = ollama.chat(model=MODEL, messages=messages, options={"temperature": 0})
    print(final.message.content)
else:
    print("The model did not use the tool:", response.message.content)

print("Exact result:", 4827 * 3916)

### Demonstration: without a tool, then with two tools 🧰

1. The same question **without a tool**: the model guesses the digits one by one.
2. We add a second tool, `today()`, which returns today's date, and let the model choose.
3. The `TOOLS` dictionary contains the **only** allowed functions. An unknown tool is refused.

In [ ]:
from datetime import date

# 1. Without a tool: the model guesses the digits one by one
print(ask([{"role": "user", "content": "How much is 4827 multiplied by 3916? Answer with the number only."}], temperature=0))

# 2. Two tools
def today():
    return date.today().isoformat()

today_description = {
    "type": "function",
    "function": {
        "name": "today",
        "description": "Return the date of today",
        "parameters": {"type": "object", "properties": {}},
    },
}

TOOLS = {"multiply": multiply, "today": today}  # 3. the only functions the model may use

def ask_with_tools(question):
    messages = [{"role": "user", "content": question}]
    response = ollama.chat(model=MODEL, messages=messages, tools=[multiply_description, today_description], options={"temperature": 0})
    if not response.message.tool_calls:
        return response.message.content
    messages.append(response.message)
    for call in response.message.tool_calls:
        name, arguments = call.function.name, call.function.arguments
        if name in TOOLS:
            try:
                result = TOOLS[name](**arguments)
            except TypeError as error:          # wrong arguments
                result = f"Error: {error}"
        else:
            result = f"Error: unknown tool {name}"
        print("  tool:", name, dict(arguments), "->", result)
        messages.append({"role": "tool", "tool_name": name, "content": str(result)})
    return ollama.chat(model=MODEL, messages=messages, options={"temperature": 0}).message.content

print(ask_with_tools("What is the date today?"))
print(ask_with_tools("How much is 12 times 34?"))

The model writes the name of the tool: it can make mistakes, or be manipulated (see TD 6). Never run a function only because the model asks for it.

> **What about FunctionGemma?** [`functiongemma:270m`](https://ollama.com/library/functiongemma) is a tiny model (0.3 GB) specialised in function calling. It is not made for chatting, and it needs very precise descriptions.

## Local or hosted? ⚖️

| | Local model | Hosted model |
| --- | --- | --- |
| **Data** | stays on your machine | goes to a provider |
| **Quality** | limited by your machine | the best models |
| **Cost** | hardware and electricity | per token |
| **Stability** | the model does not change | it may change or disappear |
| **Offline** | yes | no |

In TD 6, we will put the tools in a **loop**: the model will become an agent.